In [1]:
"""
This result works with jobs submitted by the decorenom_ibm script.
It is used to retrieve results from IBM, collect them together into collector arrays,
then save the product npy files for plotting and other analysis.
"""

'\nThis result works with jobs submitted by the decorenom_ibm script.\nIt is used to retrieve results from IBM, collect them together into collector arrays,\nthen save the product npy files for plotting and other analysis.\n'

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from backend import get_sampler
import numpy as np

In [3]:
import sys
sys.path.append('../utils/')

from RunTracker import RunTracker

from JobResult import JobResult

from DecoherenceRenormalizer import DecoherenceRenormalizer

from Collector import Collector

In [ ]:
run_track_save_path = "../sessions"

run_name = 'test'
savepath = 'results'


# If you are using this script, you probably want this to be True.
SAVEFLAG = True

rt = RunTracker()
rt.load(run_track_save_path, run_name)

print("Loaded a total of {} runs".format(len(rt)))

RunTracker initialized.
Tracked jobs loaded from ../sessions/test.json
Loaded a total of 22 runs


In [5]:
# First check if all the runs have finished
service = QiskitRuntimeService()

for run in rt:
    j = JobResult(service, run)
    if j.check_job_status() != 'Completed':
        print("Run {} is not completed yet or failed, status: {}".format(run.name, j.check_job_status()))

RuntimeJobNotFound: 'Job not found: 404 Client Error: Not Found for url: https://us-east.quantum-computing.cloud.ibm.com/jobs/c0afcb90-5bf4-4302-a3a0-a1570df7c684. {"errors":[{"code":1291,"message":"Job not found. Job ID: c0afcb90-5bf4-4302-a3a0-a1570df7c684","solution":"Verify the job ID is correct and that you have the correct access permissions.","more_info":"https://cloud.ibm.com/apidocs/quantum-computing#error-handling"}],"trace":"de631af2-3129-40d3-ac8a-2ef31cfd09d1"}\n'

In [ ]:
cl = Collector(run_name)

# We will have the the times t from the job submission run
# Here we will be collecting everything else

In [ ]:
from gc import collect


for t_c in range(len(rt) / 2): # half is estimate, half is actual

    print("####################")
    print("Running for the {}th run".format(t_c))

    job_id_identity = rt.get_job_id("identity", t_c)

    j = JobResult(service, job_id_identity)

    j.load_job_from_id()

    c = j.get_counts()

    print("Succesfullly retrieved counts for run {}: {}".format(job_id_identity, c))

    dr = DecoherenceRenormalizer(None, True)

    # esimates error rate by the number of types counts is not zero
    # @TODO: maybe we will change how we estiamte error in future
    dr.estimate_error_rate_from_counts(c)

    print("Estimated error rate:", dr.get_rate_estimate())
    cl.add("error_rate", dr.get_rate_estimate())

    job_id_circuit = rt.get_job_id("circuit", t_c)

    j = JobResult(service, job_id_circuit)

    j.load_job_from_id()

    c = j.get_counts()
    print("Succesfullly retrieved counts for run {}: {}".format(job_id_circuit, c))

    target_state = rt.get_target_state("circuit", t_c)

    # Perform physical correction, removing '11' states

    norm = 0

    for key in c.keys():
        for digi in range(N):
            if key[2*digi] == '1' and key[2*digi+1] == '1':
                c[key] = 0
                break

    for key in c.keys():
        norm += c[key]

    prob = c.get(target_state, 0) / norm
    # print("Measurement counts:", counts)
    print("Raw probability:", prob)
    prob = dr.renormalize(prob)
    print("Renormalized probability:", prob)
    cl.add("p", prob)
    staterror = np.sqrt(prob * (1 - prob) / norm)
    cl.add("staterror", staterror)

if SAVEFLAG:
    cl.save(path=savepath)
    